# 05 — Classi rare: numerosità ed errori ricorrenti

T10/T11. Leggiamo il training originale e le predizioni del notebook 04.
Nessun nuovo fit, riga eliminata o etichetta corretta. La diagnosi riguarda U2R/R2L:
quali tipologie sono poche, quali record sono sbagliati spesso e dove finiscono gli errori.
Le posizioni originali permettono di ricostruire i conteggi, senza confondere
ripetizioni dello stesso record con nuovi esempi indipendenti.


In [ ]:
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER


## 0. Ricreare il riferimento locale senza addestrare

Lo script `scripts/freeze_validation_reference.py` è codice del progetto.
Legge i risultati e le predizioni del notebook 04, verifica indici e risposte e
ricalcola metriche, matrici medie e record distinti in una cartella separata.
Si può rieseguire: non modifica i file04 e non esegue fit.


In [ ]:
import subprocess

snapshot_run = subprocess.run(
    [sys.executable, str(project_root / 'scripts/freeze_validation_reference.py'),
     '--project-root', str(project_root)],
    cwd=project_root, check=True,
)


## 1. Profilo del training e delle tipologie rare

`value_counts` conta le famiglie; `groupby` distingue le tipologie grezze.
Il fatto che una tipologia sia rara è un'osservazione: non dimostra da solo
la causa degli errori e non autorizza a cambiare le sue etichette.


In [ ]:
train_path = project_root / 'data/raw/KDDTrain+.csv'
string_types = {column: 'string' for column in (*CATEGORICAL_COLUMNS, 'attack_label')}
data = pd.read_csv(train_path, header=None, names=COLUMN_NAMES, dtype=string_types)
unknown = data['attack_label'].isna() | ~data['attack_label'].isin(ATTACK_TO_FAMILY)
if unknown.any():
    raise ValueError('Etichette mancanti o non mappate: riesaminare il target.')
y = data['attack_label'].map(ATTACK_TO_FAMILY).rename('family')
X = data.loc[:, list(FEATURE_COLUMNS)].copy()
assert X.index.equals(pd.RangeIndex(len(X))) and X.shape[1] == 41
reference_path = project_root / 'artifacts/rare_class_reference'
reference = json.loads((reference_path / 'results.json').read_text(encoding='utf-8'))
assert hashlib.sha256(train_path.read_bytes()).hexdigest() == reference['source_sha256']
assert reference['seeds'] == list(range(1, 6)) and reference['feature_count'] == 41
rare_families = ['R2L', 'U2R']
family_counts = y.value_counts().reindex(FAMILY_ORDER)
raw_labels = data[['attack_label']].assign(family=y)
type_counts = raw_labels.loc[y.isin(rare_families)].groupby(
    ['family', 'attack_label'], observed=True,
).size().rename('training_records').reset_index()
display(family_counts.rename('record nel training'))
display(type_counts)
assert family_counts['U2R'] == 52 and family_counts['R2L'] == 995
assert type_counts.groupby('family')['training_records'].sum().to_dict() == {'R2L': 995, 'U2R': 52}


## 2. Ricostruire esposizioni e predizioni dai file salvati

Per ogni run controlliamo le risposte vere rispetto alle posizioni del CSV.
Conserviamo gli eventi con famiglia vera rara o previsione rara: così possiamo
leggere sia attacchi persi sia falsi positivi. Il supporto del train è il totale
meno il supporto della validation; le tre profondità della stessa run condividono le righe.


In [ ]:
event_frames = []
support_rows = []
with np.load(reference_path / 'predictions.npz', allow_pickle=False) as archive:
    for run, seed in enumerate(reference['seeds'], start=1):
        indices = archive[f'run_{run:02d}_validation_indices']
        truth = archive[f'run_{run:02d}_truth']
        np.testing.assert_array_equal(truth, y.iloc[indices].to_numpy())
        validation_counts = pd.Series(truth).value_counts().reindex(FAMILY_ORDER, fill_value=0)
        for family in FAMILY_ORDER:
            support_rows.append({
                'run': run, 'seed': seed, 'family': family,
                'train_records': int(family_counts[family] - validation_counts[family]),
                'validation_records': int(validation_counts[family]),
            })
        for depth in reference['depths']:
            predicted = archive[f'run_{run:02d}_depth_{depth}']
            keep = np.isin(truth, rare_families) | np.isin(predicted, rare_families)
            selected = indices[keep]
            frame = pd.DataFrame({
                'row_position': selected, 'run': run, 'seed': seed, 'max_depth': depth,
                'family': truth[keep], 'predicted': predicted[keep],
                'attack_label': data.iloc[selected]['attack_label'].to_numpy(),
            })
            frame['is_error'] = frame['family'].ne(frame['predicted'])
            event_frames.append(frame)

events = pd.concat(event_frames, ignore_index=True)
rare_events = events.loc[events['family'].isin(rare_families)].copy()
split_support = pd.DataFrame(support_rows)
display(split_support.groupby('family')[['train_records', 'validation_records']].agg(['min', 'max']))


## 3. Frequenza di errore di un record

Un record apparso due volte e sbagliato una volta ha frequenza di errore **1/2**,
non 1/5. Raggruppiamo separatamente ogni profondità e posizione originale.
I conteggi `predicted_...` dicono quali risposte ha ricevuto nelle sue apparizioni.


In [ ]:
record_keys = ['max_depth', 'row_position', 'family', 'attack_label']
record_diagnosis = rare_events.groupby(record_keys, sort=False).agg(
    n_validation=('run', 'size'), n_errors=('is_error', 'sum'),
).reset_index()
record_diagnosis['error_rate'] = record_diagnosis['n_errors'] / record_diagnosis['n_validation']
predicted_counts = pd.crosstab(
    [rare_events[key] for key in record_keys], rare_events['predicted'],
).reindex(columns=FAMILY_ORDER, fill_value=0).add_prefix('predicted_').reset_index()
record_diagnosis = record_diagnosis.merge(predicted_counts, on=record_keys, validate='one_to_one')


In [ ]:
print('U2R più spesso sbagliati a profondità 15 (fino a 15 record):')
display(record_diagnosis.loc[
    record_diagnosis['max_depth'].eq(15) & record_diagnosis['family'].eq('U2R')
].sort_values(['error_rate', 'n_validation'], ascending=False).head(15))
unique_observed = rare_events['row_position'].unique()
rare_indices = data.index[y.isin(rare_families)].to_numpy()
never_indices = np.setdiff1d(rare_indices, unique_observed)
never_validated = data.iloc[never_indices][['attack_label']].assign(
    row_position=never_indices, family=y.iloc[never_indices].to_numpy(),
)
display(never_validated.groupby('family').size().rename('record mai in validation'))
assert record_diagnosis['n_errors'].le(record_diagnosis['n_validation']).all()
assert record_diagnosis['n_validation'].between(1, len(reference['seeds'])).all()


## 4. Aggregare per tipologia e distinguere falsi positivi

La tabella per tipologia usa come denominatore le apparizioni effettive in validation,
riutilizzando alcuni record. `training_records` e `distinct_validation_records`
mostrano quanti esempi distinti stanno dietro alle percentuali.
La seconda tabella descrive gli errori che vengono **predetti** come R2L/U2R,
anche quando la famiglia vera è diversa.


In [ ]:
type_diagnosis = rare_events.groupby(['max_depth', 'family', 'attack_label']).agg(
    n_validation=('run', 'size'), n_errors=('is_error', 'sum'),
    distinct_validation_records=('row_position', 'nunique'),
).reset_index()
type_diagnosis['error_rate'] = type_diagnosis['n_errors'] / type_diagnosis['n_validation']
type_diagnosis = type_diagnosis.merge(type_counts, on=['family', 'attack_label'], validate='many_to_one')
false_positive_events = events.loc[
    events['predicted'].isin(rare_families) & events['is_error']
]
false_positive_types = false_positive_events.groupby(
    ['max_depth', 'predicted', 'family', 'attack_label'],
).agg(occurrences=('run', 'size'), distinct_records=('row_position', 'nunique')).reset_index()
display(type_diagnosis.loc[type_diagnosis['family'].eq('U2R')].round(4))
display(false_positive_types.loc[false_positive_types['max_depth'].eq(15)].sort_values('occurrences', ascending=False).head(12))

# Riconciliazione con i conteggi delle matrici di ogni run.
for depth in reference['depths']:
    for i, family in enumerate(FAMILY_ORDER):
        if family in rare_families:
            counts = np.asarray(reference['confusion_counts_by_depth'][str(depth)])
            expected_errors = int(counts[:, i, :].sum() - counts[:, i, i].sum())
            observed_errors = int(rare_events.loc[
                rare_events['max_depth'].eq(depth) & rare_events['family'].eq(family), 'is_error'
            ].sum())
            assert observed_errors == expected_errors


## 5. Salvare la diagnosi senza cambiare i dati

Le tabelle dettagliate restano locali. L'analisi di errori della validation orienta
ipotesi e scelte di sviluppo: non diventa una valutazione indipendente, e non
autorizza a correggere etichette o eliminare record difficili.


In [ ]:
artifacts = project_root / 'artifacts/rare_class_diagnosis'
artifacts.mkdir(parents=True, exist_ok=True)
type_counts.to_csv(artifacts / 'type_counts.csv', index=False)
split_support.to_csv(artifacts / 'split_support.csv', index=False)
events.to_csv(artifacts / 'evaluation_events.csv', index=False)
record_diagnosis.to_csv(artifacts / 'record_diagnosis.csv', index=False)
type_diagnosis.to_csv(artifacts / 'type_diagnosis.csv', index=False)
false_positive_types.to_csv(artifacts / 'false_positive_types.csv', index=False)
never_validated.to_csv(artifacts / 'never_validated.csv', index=False)
observed_counts = record_diagnosis.loc[record_diagnosis['max_depth'].eq(15)].groupby('family').size()
assert observed_counts['U2R'] == reference['unique_validation_counts']['U2R']
print('Diagnosi salvata; nessun fit o modifica ai CSV originali.')


## Lettura degli esiti

L'errore ricorrente di un record non prova che l'etichetta sia sbagliata.
Per le tipologie con pochi casi, percentuali elevate possono dipendere da pochissime
osservazioni. I record mai in validation non hanno un tasso di errore osservato.
Il prossimo intervento concordato è la pesatura delle classi nel fit, non il filtraggio.
Sintesi in [RARE_CLASS_DIAGNOSIS.md](../docs/RARE_CLASS_DIAGNOSIS.md).


## Esito verificato

Nei cinque split compaiono31 U2R distinti. A profondità15, rootkit è sbagliato
in12/13 apparizioni e loadmodule in7/8; perl è corretto in2/2, troppo pochi per
una conclusione generale.18 U2R distinti sono sempre sbagliati nelle loro apparizioni.
Le osservazioni non giustificano correzioni delle etichette o filtri sui dati.
[Diagnosi e limiti](../docs/RARE_CLASS_DIAGNOSIS.md).
